In [ ]:
# Senatör Featureunda yapmak istediğim:
# Senatörlerin rankingini sağlamak bu 2 farklı yolun birleşmesiyle olucak
# Biri senator trust index diyebiliriz bu geçmişe yönelik işlemlerde
# Bunda 2 spektrum yaparız buy farklı sell farklı olur 
# Transaction date ve received datedeki fiyatları karşılaştıracak
# Bu sayede aradaki sürede fiyat farkı az mı olmuş uzun vadeli mi
# Kısa vadeli mi yatırım yapıyor görüyor olacağız
# İkincisi senatör performans endeksi backtestle beraber
# Bir nevi senatörlerin performanslarını yorumlayacağız
# Bu elde ettiğimiz rankingi kullanacğımız bir faz olucak
# Ekonomik metriklerle sıraladığımız ve belli bir değerin üstünü yatırım
# için değerlendirmeye geçmeden yakın zamanda senatörler kendi
# rating katsayılarıyla beraber buy pozitif etki edicek
# sell negatif etki edicek sonrasında yine ranking hisselerimiz olacak

# Ayrıca yapılacaklar: 
# Otomatik power bi raporları
# JupyterHUBA geçiş ya da benzeri
# Trade box enjeksityonu
# Zipline a bak https://zipline.ml4trading.io


# Backtest'e liquide e geçme




# Ranking






# Senatör trust index 
# Senatör performans endeksi 
# ETF lerin ticker tutma
# ETF lerin sektör
# ETF için detaylı döküman hazırla Yağmur 
# Fraud Detection



# Data Cleaning Section

In [3]:
import yfinance
import pandas as pd
import numpy as np

# Gerekli veriyi çekiyoruz
df = pd.read_csv("all_companies_balance_statements.csv")
df2 = pd.read_csv("all_companies_cashflow_statements.csv")
df3 = pd.read_csv("all_companies_income_statements.csv")
df4 = pd.read_excel("listof90year.xlsx")

# Prefixler ekleyerek verilerin ne olduğunu ekliyoruz
df_bs = df.add_prefix('bs_')
df_cf = df2.add_prefix('cf_')
df_is = df3.add_prefix('is_')
for col in ['symbol', 'calendarYear', 'period']:
    df_bs[col] = df[col]
    df_cf[col] = df2[col]
    df_is[col] = df3[col]

# Join key'leri ekleyip mergelüyoruz
combined_df = df_bs.merge(df_cf, on=['symbol', 'calendarYear', 'period'], how='outer')
combined_df = combined_df.merge(df_is, on=['symbol', 'calendarYear', 'period'], how='outer')

# combined_dfdeki boş rowları atıyorum ("bs_Symbol"'e göre)
cleaned_df = combined_df[combined_df['bs_Symbol'].notnull()]

In [4]:
# Hala boş valueları olanlar neler onu kontrol ediyorum
import pandas as pd
from tabulate import tabulate

missing_summary = cleaned_df.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
missing_df = pd.DataFrame({
    "Column": missing_summary.index,
    "Missing Values": missing_summary.values,
    "Missing Rate (%)": (missing_summary.values / len(cleaned_df)) * 100
})
print(tabulate(missing_df, headers='keys', tablefmt='pretty'))
missing_columns = missing_df["Column"].tolist()
with open("missing_columns.txt", "w") as f:
    for col in missing_columns:
        f.write(col + "\n")
with open("columns.txt", "w") as f:
    for col in cleaned_df.columns:
        f.write(col + "\n")

+-----+---------------------------------------------+----------------+-----------------------+
|     |                   Column                    | Missing Values |   Missing Rate (%)    |
+-----+---------------------------------------------+----------------+-----------------------+
|  0  |                is_finalLink                 |     47318      |  31.528308046987956   |
|  1  |                   is_link                   |     47313      |   31.52497651268315   |
|  2  |                cf_finalLink                 |     47182      |   31.43769031389716   |
|  3  |                   cf_link                   |     47182      |   31.43769031389716   |
|  4  |                   bs_link                   |     47102      |  31.384385765020223   |
|  5  |                bs_finalLink                 |     47102      |  31.384385765020223   |
|  6  |                   cf_cik                    |      4478      |  2.9837221233867046   |
|  7  |        cf_effectOfForexChangesOnCash      

In [5]:
# Linklere metriklerde ihtiyaç olmayacağı için çıkarıp tekrar hataları çıkarıyorum
drop_cols = [
    "is_finalLink", "is_link",
    "cf_finalLink", "cf_link",
    "bs_link", "bs_finalLink"
]
filtered_missing_df = missing_df[~missing_df["Column"].isin(drop_cols)]
from tabulate import tabulate
print(tabulate(filtered_missing_df, headers='keys', tablefmt='pretty'))
filtered_columns = filtered_missing_df["Column"].tolist()
with open("filtered_missing_columns.txt", "w") as f:
    for col in filtered_columns:
        f.write(col + "\n")

+-----+---------------------------------------------+----------------+-----------------------+
|     |                   Column                    | Missing Values |   Missing Rate (%)    |
+-----+---------------------------------------------+----------------+-----------------------+
|  6  |                   cf_cik                    |      4478      |  2.9837221233867046   |
|  7  |        cf_effectOfForexChangesOnCash        |      4341      |   2.892438083434945   |
|  8  |          cf_stockBasedCompensation          |      4341      |   2.892438083434945   |
|  9  |             cf_netChangeInCash              |      4340      |  2.8917717765739837   |
| 10  |             cf_acquisitionsNet              |      4340      |  2.8917717765739837   |
| 11  |            cf_deferredIncomeTax             |      4340      |  2.8917717765739837   |
| 12  |          cf_commonStockRepurchased          |      4340      |  2.8917717765739837   |
| 13  |          cf_purchasesOfInvestments        

In [6]:
# CIK numarası dolduktan sonra nerelerde eksik olduğunu inceliyoruz
symbol_columns = [col for col in filtered_columns if "symbol" in col.lower()]
cik_columns = [col for col in filtered_columns if "cik" in col.lower()]
for col in symbol_columns:
    cleaned_df[col] = cleaned_df["is_Symbol"]
for col in cik_columns:
    cleaned_df[col] = cleaned_df["is_cik"]
missing_summary = cleaned_df.isnull().sum()
missing_summary = missing_summary[missing_summary > 0]
filtered_missing_df_updated = (
    pd.DataFrame({
        "Column": missing_summary.index,
        "Missing Values": missing_summary.values
    })
    .query("Column not in @drop_cols")
    .sort_values("Missing Values", ascending=False)
)
filtered_missing_df_updated["Missing Rate (%)"] = (
    filtered_missing_df_updated["Missing Values"] / len(cleaned_df) * 100
)
print(tabulate(filtered_missing_df_updated, headers='keys', tablefmt='pretty'))
with open("filtered_missing_columns.txt", "w") as f:
    for col in filtered_missing_df_updated["Column"]:
        f.write(col + "\n")

+-----+---------------------------------------------+----------------+-----------------------+
|     |                   Column                    | Missing Values |   Missing Rate (%)    |
+-----+---------------------------------------------+----------------+-----------------------+
| 36  |          cf_stockBasedCompensation          |      4341      |   2.892438083434945   |
| 56  |        cf_effectOfForexChangesOnCash        |      4341      |   2.892438083434945   |
| 57  |             cf_netChangeInCash              |      4340      |  2.8917717765739837   |
| 35  |            cf_deferredIncomeTax             |      4340      |  2.8917717765739837   |
| 45  |             cf_acquisitionsNet              |      4340      |  2.8917717765739837   |
| 52  |          cf_commonStockRepurchased          |      4340      |  2.8917717765739837   |
| 46  |          cf_purchasesOfInvestments          |      4339      |   2.891105469713022   |
| 47  |       cf_salesMaturitiesOfInvestments     

/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_1116/1529254276.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df[col] = cleaned_df["is_Symbol"]
/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_1116/1529254276.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df[col] = cleaned_df["is_cik"]


In [7]:
# Eğer boş hücresi varsa o satırı silen kod
columns_to_check = filtered_missing_df_updated["Column"].tolist()
cleaned_df.dropna(subset=columns_to_check, inplace=True)

/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_1116/327273373.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df.dropna(subset=columns_to_check, inplace=True)


In [8]:
# Bu sayede cleaned_dfdeki veri kayıpları önemsiz olucak
unchanged_df = cleaned_df

# Data Manipulation and Necessary Data Injection

## CIK Numbers - Ticker Matching

### Tickers with More Than One Associated CIK Number

In [11]:
# Symbol başına CIK sayısını kontrol ediyoruz
cik_per_symbol = cleaned_df.groupby("symbol")["bs_cik"].nunique()
symbols_with_multiple_ciks = cik_per_symbol[cik_per_symbol > 1]
print(f"Birden fazla CIK'e sahip symbol sayısı: {len(symbols_with_multiple_ciks)}")
print(symbols_with_multiple_ciks.sort_values(ascending=False).head(45))

Birden fazla CIK'e sahip symbol sayısı: 45
symbol
ARRKF    2
ICTSF    2
IPHA     2
IRET     2
NBLWF    2
NE       2
NEWP     2
NUTR     2
NVAAF    2
OILCF    2
PAPL     2
PUBGY    2
QIPT     2
RR       2
SSL      2
TELO     2
TLN      2
TLSA     2
TSM      2
VSTS     2
WAY      2
INHD     2
IBACR    2
ASML     2
IBAC     2
BCH      2
BDC      2
BIDU     2
BKHA     2
BKHAR    2
BKHAU    2
BLK      2
BSAC     2
COOTW    2
CP       2
DRUG     2
EBBNF    2
EBGEF    2
ENB      2
ENGN     2
ENGNW    2
ENJ      2
GEV      2
GOVB     2
XPER     2
Name: bs_cik, dtype: int64


In [12]:
# Bir sembol birden fazla CIK'e sahipse, hangi CIK'ler olduğunu görelim
multi_cik_symbols = cik_per_symbol[cik_per_symbol > 1].index
for symbol in multi_cik_symbols:
    ciks = cleaned_df[cleaned_df["symbol"] == symbol]["bs_cik"].unique()
    print(f"{symbol}: {list(ciks)}")

ARRKF: [1855743.0, 0.0]
ASML: [0.0, 937966.0]
BCH: [0.0, 1161125.0]
BDC: [913142.0, 1896212.0]
BIDU: [0.0, 1329099.0]
BKHA: [0.0, 2000775.0]
BKHAR: [0.0, 2000775.0]
BKHAU: [0.0, 2000775.0]
BLK: [1364742.0, 2012383.0]
BSAC: [0.0, 1027552.0]
COOTW: [0.0, 1959994.0]
CP: [0.0, 16875.0]
DRUG: [0.0, 1827401.0]
EBBNF: [0.0, 895728.0]
EBGEF: [0.0, 895728.0]
ENB: [0.0, 895728.0]
ENGN: [0.0, 1100663.0]
ENGNW: [0.0, 1100663.0]
ENJ: [65984.0, 71508.0]
GEV: [0.0, 1996810.0]
GOVB: [0.0, 1978811.0]
IBAC: [0.0, 1998781.0]
IBACR: [0.0, 1998781.0]
ICTSF: [1010134.0, 0.0]
INHD: [0.0, 1961847.0]
IPHA: [0.0, 1598599.0]
IRET: [0.0, 798359.0]
NBLWF: [1895262.0, 1458891.0]
NE: [1895262.0, 1458891.0]
NEWP: [0.0, 1369085.0]
NUTR: [0.0, 105007.0]
NVAAF: [0.0, 1852551.0]
OILCF: [0.0, 1922639.0]
PAPL: [0.0, 1938109.0]
PUBGY: [1050952.0, 0.0]
QIPT: [0.0, 1540013.0]
RR: [0.0, 1963685.0]
SSL: [0.0, 314590.0]
TELO: [0.0, 1971532.0]
TLN: [0.0, 1622536.0]
TLSA: [0.0, 1723069.0]
TSM: [0.0, 1046179.0]
VSTS: [0.0, 1967649.

In [13]:
# 0.0 olanları diğerine eşitliyoruz
zero_cik_mask = cleaned_df["bs_cik"] == 0.0
symbols_with_zero_cik = cleaned_df[zero_cik_mask]["symbol"].unique()
for symbol in symbols_with_zero_cik:
    valid_ciks = cleaned_df[(cleaned_df["symbol"] == symbol) & (cleaned_df["bs_cik"] != 0.0)]["bs_cik"].unique()
    if len(valid_ciks) == 1:
        cleaned_df.loc[(cleaned_df["symbol"] == symbol) & (cleaned_df["bs_cik"] == 0.0), "bs_cik"] = valid_ciks[0]
remaining_zeros = cleaned_df[cleaned_df["bs_cik"] == 0.0]

In [14]:
# Symbol başına CIK sayısını tekrar kontrol ediyoruz
cik_per_symbol = cleaned_df.groupby("symbol")["bs_cik"].nunique()
symbols_with_multiple_ciks = cik_per_symbol[cik_per_symbol > 1]
multi_cik_symbols = cik_per_symbol[cik_per_symbol > 1].index
for symbol in multi_cik_symbols:
    ciks = cleaned_df[cleaned_df["symbol"] == symbol]["bs_cik"].unique()
    print(f"{symbol}: {list(ciks)}")

BDC: [913142.0, 1896212.0]
BLK: [1364742.0, 2012383.0]
ENJ: [65984.0, 71508.0]
NBLWF: [1895262.0, 1458891.0]
NE: [1895262.0, 1458891.0]
XPER: [1788999.0, 1803696.0]


In [15]:
# Bu sorunu ortadan kaldırmak için FMP'den yardım alıyoruz
"""
try:
    from urllib.request import urlopen
except ImportError:
    from urllib2 import urlopen

import certifi
import json
import pandas as pd

def read_api_key_from_file(file_path):
    with open(file_path, "r") as file:
        return file.read().strip()

def get_jsonparsed_data(url):
    try:
        response = urlopen(url, cafile=certifi.where())
        data = response.read().decode("utf-8")
        return json.loads(data)
    except Exception as e:
        print(f"❌ Bağlantı hatası: {e}")
        return None

def normalize_cik(cik):
    try:
        return str(int(float(cik))).zfill(10)
    except:
        return None

# 1. API key al
api_key = read_api_key_from_file("FMP API KEY.txt")

# 2. Symbol başına birden fazla CIK olup olmadığını kontrol et
cik_per_symbol = cleaned_df.groupby("symbol")["bs_cik"].nunique()
multi_cik_symbols = cik_per_symbol[cik_per_symbol > 1].index

# 3. FMP üzerinden kontrol
for symbol in multi_cik_symbols:
    ciks = cleaned_df[cleaned_df["symbol"] == symbol]["bs_cik"].dropna().unique()
    for cik in ciks:
        cik_str = normalize_cik(cik)
        if cik_str is None or cik_str == "0000000000":
            continue  # Geçersiz cik atla

        url = f"https://financialmodelingprep.com/api/v3/search-cik?cik={cik_str}&apikey={api_key}"
        response = get_jsonparsed_data(url)

        if response:
            match_found = False
            for item in response:
                fmp_symbol = item.get("symbol")
                if fmp_symbol == symbol:
                    print(f"✅ UYUMLU: {symbol} == {fmp_symbol}, cik={cik_str}")
                    match_found = True
                    break
            if not match_found:
                print(f"⚠️ UYUŞMADI: Yerel symbol={symbol}, gelenler={[r.get('symbol') for r in response]}, cik={cik_str}")
        else:
            print(f"⚠️ CIK için sonuç yok: cik={cik_str}, symbol={symbol}")
"""

'\ntry:\n    from urllib.request import urlopen\nexcept ImportError:\n    from urllib2 import urlopen\n\nimport certifi\nimport json\nimport pandas as pd\n\ndef read_api_key_from_file(file_path):\n    with open(file_path, "r") as file:\n        return file.read().strip()\n\ndef get_jsonparsed_data(url):\n    try:\n        response = urlopen(url, cafile=certifi.where())\n        data = response.read().decode("utf-8")\n        return json.loads(data)\n    except Exception as e:\n        print(f"❌ Bağlantı hatası: {e}")\n        return None\n\ndef normalize_cik(cik):\n    try:\n        return str(int(float(cik))).zfill(10)\n    except:\n        return None\n\n# 1. API key al\napi_key = read_api_key_from_file("FMP API KEY.txt")\n\n# 2. Symbol başına birden fazla CIK olup olmadığını kontrol et\ncik_per_symbol = cleaned_df.groupby("symbol")["bs_cik"].nunique()\nmulti_cik_symbols = cik_per_symbol[cik_per_symbol > 1].index\n\n# 3. FMP üzerinden kontrol\nfor symbol in multi_cik_symbols:\n    ci

### CIK Numbers  with More Than One Associated Ticker


In [16]:
# Bu sorunu ortadan kaldırmak için FMP'den yardım alıyoruz
"""
try:
    from urllib.request import urlopen
except ImportError:
    from urllib2 import urlopen

import certifi
import json
import pandas as pd

def read_api_key_from_file(file_path):
    with open(file_path, "r") as file:
        return file.read().strip()

def get_jsonparsed_data(url):
    try:
        response = urlopen(url, cafile=certifi.where())
        data = response.read().decode("utf-8")
        return json.loads(data)
    except Exception as e:
        print(f"❌ Bağlantı hatası: {e}")
        return None

def normalize_cik(cik):
    try:
        return str(int(float(cik))).zfill(10)
    except:
        return None

# 1. API key al
api_key = read_api_key_from_file("FMP API KEY.txt")

# 2. Symbol başına birden fazla CIK olup olmadığını kontrol et
cik_per_symbol = cleaned_df.groupby("symbol")["bs_cik"].nunique()
multi_cik_symbols = cik_per_symbol[cik_per_symbol > 1].index

# 3. FMP üzerinden kontrol
for symbol in multi_cik_symbols:
    ciks = cleaned_df[cleaned_df["symbol"] == symbol]["bs_cik"].dropna().unique()
    for cik in ciks:
        cik_str = normalize_cik(cik)
        if cik_str is None or cik_str == "0000000000":
            continue  # Geçersiz cik atla

        url = f"https://financialmodelingprep.com/api/v3/search-cik?cik={cik_str}&apikey={api_key}"
        response = get_jsonparsed_data(url)

        if response:
            match_found = False
            for item in response:
                fmp_symbol = item.get("symbol")
                if fmp_symbol == symbol:
                    print(f"✅ UYUMLU: {symbol} == {fmp_symbol}, cik={cik_str}")
                    match_found = True
                    break
            if not match_found:
                print(f"⚠️ UYUŞMADI: Yerel symbol={symbol}, gelenler={[r.get('symbol') for r in response]}, cik={cik_str}")
        else:
            print(f"⚠️ CIK için sonuç yok: cik={cik_str}, symbol={symbol}")
"""

'\ntry:\n    from urllib.request import urlopen\nexcept ImportError:\n    from urllib2 import urlopen\n\nimport certifi\nimport json\nimport pandas as pd\n\ndef read_api_key_from_file(file_path):\n    with open(file_path, "r") as file:\n        return file.read().strip()\n\ndef get_jsonparsed_data(url):\n    try:\n        response = urlopen(url, cafile=certifi.where())\n        data = response.read().decode("utf-8")\n        return json.loads(data)\n    except Exception as e:\n        print(f"❌ Bağlantı hatası: {e}")\n        return None\n\ndef normalize_cik(cik):\n    try:\n        return str(int(float(cik))).zfill(10)\n    except:\n        return None\n\n# 1. API key al\napi_key = read_api_key_from_file("FMP API KEY.txt")\n\n# 2. Symbol başına birden fazla CIK olup olmadığını kontrol et\ncik_per_symbol = cleaned_df.groupby("symbol")["bs_cik"].nunique()\nmulti_cik_symbols = cik_per_symbol[cik_per_symbol > 1].index\n\n# 3. FMP üzerinden kontrol\nfor symbol in multi_cik_symbols:\n    ci

In [17]:
# CIK ve Symbollerin sayılarını kontrol ediyorum
symbol_per_cik = cleaned_df.groupby("bs_cik")["symbol"].nunique()
ciks_with_multiple_symbols = symbol_per_cik[symbol_per_cik > 1]
print(f"Birden fazla symbol'e sahip cik sayısı: {len(ciks_with_multiple_symbols)}")
print(ciks_with_multiple_symbols.sort_values(ascending=False).head(10))  # ilk 10 örnek

Birden fazla symbol'e sahip cik sayısı: 762
bs_cik
0.0          892
1026214.0     25
70858.0       17
310522.0      16
1393311.0     15
23426.0       13
1002910.0     12
895421.0      11
1464790.0      9
1004980.0      9
Name: symbol, dtype: int64


In [18]:
# 0.0 CIK numarasına sahip olanları inceleyelim
unique_symbols_with_zero_cik = cleaned_df[cleaned_df["bs_cik"] == 0.0]["bs_Symbol"].dropna().unique()
print(f"0.0 bs_cik değerine sahip unique bs_Symbol sayısı: {len(unique_symbols_with_zero_cik)}")
print(unique_symbols_with_zero_cik)

0.0 bs_cik değerine sahip unique bs_Symbol sayısı: 892
['AAMTF' 'AAUGF' 'ABAT' 'ABBNY' 'ABVE' 'ABVEW' 'ABVX' 'ABXXF' 'ACPS'
 'ACRL' 'ACZT' 'ADGM' 'ADIA' 'AERGP' 'AERS' 'AFJK' 'AFJKR' 'AFJKU' 'AFLYY'
 'AFRAF' 'AGQPF' 'AHL-PD' 'AHL-PE' 'AHRO' 'AIEV' 'AIJTY' 'AIOT' 'AIQUF'
 'AIQUY' 'AIRE' 'AIRJ' 'AIRJW' 'AITR' 'AITRR' 'AITRU' 'ALAB' 'ALMS'
 'ALSMY' 'ALYAF' 'AMIX' 'AMTM' 'AMX' 'ANL' 'ANSC' 'ANSCU' 'ANSCW' 'ANYYY'
 'AOAO' 'AOMFF' 'APHP' 'AQN' 'ARBK' 'ARBKF' 'ARDT' 'ARMN' 'ARTV' 'ASII'
 'ASR' 'ASX' 'ATGL' 'ATHE' 'ATLN' 'ATMH' 'ATPC' 'ATYR' 'AU' 'AUNA' 'AVAI'
 'AVBP' 'AVRW' 'AXREF' 'AYR' 'AYRWF' 'AZN' 'AZUL' 'BAFBF' 'BAK' 'BAM'
 'BAMXF' 'BAYA' 'BAYAR' 'BAYAU' 'BBAR' 'BBVA' 'BCAL' 'BCEPF' 'BCG' 'BCGWW'
 'BCPPF' 'BCUCF' 'BECEF' 'BEEP' 'BETRF' 'BETSF' 'BEWFF' 'BGC' 'BHRB'
 'BIOA' 'BIOE' 'BIRK' 'BKAMF' 'BKFAF' 'BKFPF' 'BKGFF' 'BLFR' 'BLMH' 'BLMZ'
 'BMA' 'BNAI' 'BNAIW' 'BNIGF' 'BNPQF' 'BOLD' 'BOW' 'BP' 'BPTSY' 'BRENF'
 'BRFS' 'BROXF' 'BRQSF' 'BRRLY' 'BRSHF' 'BRSYF' 'BSBR' 'BSQKZ' 'BTGRF'
 'BTI' 'B

In [19]:
# 32 tane farklı tickera sahip olan CIK numarasını inceliyoruz
unique_symbols_with_cik = cleaned_df[cleaned_df["bs_cik"] == 1026214.0]["bs_Symbol"].dropna().unique()
print(f"0.0 bs_cik değerine sahip unique bs_Symbol sayısı: {len(unique_symbols_with_cik)}")
print(unique_symbols_with_cik)

0.0 bs_cik değerine sahip unique bs_Symbol sayısı: 25
['FMCC' 'FMCCG' 'FMCCH' 'FMCCI' 'FMCCJ' 'FMCCK' 'FMCCL' 'FMCCM' 'FMCCN'
 'FMCCO' 'FMCCP' 'FMCCS' 'FMCCT' 'FMCKI' 'FMCKJ' 'FMCKK' 'FMCKL' 'FMCKM'
 'FMCKN' 'FMCKO' 'FMCKP' 'FREGP' 'FREJN' 'FREJO' 'FREJP']


In [20]:
# 0.0 olan CIK numaralarını geçici olarak temizliyoruz
valid_cik_df = cleaned_df[cleaned_df["bs_cik"] != 0.0]
symbol_per_cik = valid_cik_df.groupby("bs_cik")["symbol"].nunique()
ciks_with_multiple_symbols = symbol_per_cik[symbol_per_cik > 1]

print(f"Birden fazla symbol'e sahip geçerli cik sayısı: {len(ciks_with_multiple_symbols)}")
print(ciks_with_multiple_symbols.sort_values(ascending=False).head(10))

Birden fazla symbol'e sahip geçerli cik sayısı: 761
bs_cik
1026214.0    25
70858.0      17
310522.0     16
1393311.0    15
23426.0      13
1002910.0    12
895421.0     11
1004980.0     9
1464790.0     9
895728.0      8
Name: symbol, dtype: int64


In [21]:
cleaned_df

,bs_Symbol,bs_Unnamed: 1,bs_date,bs_symbol,bs_reportedCurrency,bs_cik,bs_fillingDate,bs_acceptedDate,bs_calendarYear,bs_period,...,is_incomeBeforeTaxRatio,is_incomeTaxExpense,is_netIncome,is_netIncomeRatio,is_eps,is_epsdiluted,is_weightedAverageShsOut,is_weightedAverageShsOutDil,is_link,is_finalLink
0,A,26.0,1998-10-31,A,USD,1090872.0,1998-10-31,1998-10-31 00:00:00,1998.0,FY,...,0.049799,139000000.0,2.570000e+08,0.032319,0.58,0.56,443103448.0,458928571.0,NaN,NaN
1,A,25.0,1999-10-31,A,USD,1090872.0,2000-01-25,2000-01-25 00:00:00,1999.0,FY,...,0.094466,275000000.0,5.120000e+08,0.061457,1.35,1.35,380000000.0,457500000.0,https://www.sec.gov/Archives/edgar/data/109087...,https://www.sec.gov/Archives/edgar/data/109087...
2,A,24.0,2000-10-31,A,USD,1090872.0,2001-01-17,2001-01-17 00:00:00,2000.0,FY,...,0.108048,407000000.0,7.570000e+08,0.070268,1.68,1.66,449000000.0,455000000.0,https://www.sec.gov/Archives/edgar/data/109087...,https://www.sec.gov/Archives/edgar/data/109087...
3,A,23.0,2001-10-31,A,USD,1090872.0,2002-01-22,2002-01-22 00:00:00,2001.0,FY,...,-0.056813,-71000000.0,1.680000e+08,0.020010,0.38,0.38,458000000.0,458000000.0,https://www.sec.gov/Archives/edgar/data/109087...,https://www.sec.gov/Archives/edgar/data/109087...
4,A,22.0,2002-10-31,A,USD,1090872.0,2002-12-20,2002-12-20 17:27:53,2002.0,FY,...,-0.257404,-525000000.0,-1.032000e+09,-0.171714,-2.22,-2.22,465000000.0,465000000.0,https://www.sec.gov/Archives/edgar/data/109087...,https://www.sec.gov/Archives/edgar/data/109087...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153340,ZYXI,4.0,2019-12-31,ZYXI,USD,846475.0,2020-02-27,2020-02-27 17:24:43,2019.0,FY,...,0.262601,2449000.0,9.492000e+06,0.208744,0.27,0.25,35682898.0,37359298.0,https://www.sec.gov/Archives/edgar/data/846475...,https://www.sec.gov/Archives/edgar/data/846475...
153341,ZYXI,3.0,2020-12-31,ZYXI,USD,846475.0,2021-02-25,2021-02-25 17:01:35,2020.0,FY,...,0.126719,1079000.0,9.074000e+06,0.113252,0.24,0.24,37255898.0,38437298.0,https://www.sec.gov/Archives/edgar/data/846475...,https://www.sec.gov/Archives/edgar/data/846475...
153342,ZYXI,2.0,2021-12-31,ZYXI,USD,846475.0,2022-03-22,2022-03-21 19:19:49,2021.0,FY,...,0.170920,5168000.0,1.710300e+07,0.131258,0.41,0.40,42148698.0,43116698.0,https://www.sec.gov/Archives/edgar/data/846475...,https://www.sec.gov/Archives/edgar/data/846475...
153343,ZYXI,1.0,2022-12-31,ZYXI,USD,846475.0,2023-03-14,2023-03-13 18:49:43,2022.0,FY,...,0.140345,5150000.0,1.704800e+07,0.107785,0.44,0.44,38467000.0,39127000.0,https://www.sec.gov/Archives/edgar/data/846475...,https://www.sec.gov/Archives/edgar/data/846475...


In [22]:
valid_cik_df

,bs_Symbol,bs_Unnamed: 1,bs_date,bs_symbol,bs_reportedCurrency,bs_cik,bs_fillingDate,bs_acceptedDate,bs_calendarYear,bs_period,...,is_incomeBeforeTaxRatio,is_incomeTaxExpense,is_netIncome,is_netIncomeRatio,is_eps,is_epsdiluted,is_weightedAverageShsOut,is_weightedAverageShsOutDil,is_link,is_finalLink
0,A,26.0,1998-10-31,A,USD,1090872.0,1998-10-31,1998-10-31 00:00:00,1998.0,FY,...,0.049799,139000000.0,2.570000e+08,0.032319,0.58,0.56,443103448.0,458928571.0,NaN,NaN
1,A,25.0,1999-10-31,A,USD,1090872.0,2000-01-25,2000-01-25 00:00:00,1999.0,FY,...,0.094466,275000000.0,5.120000e+08,0.061457,1.35,1.35,380000000.0,457500000.0,https://www.sec.gov/Archives/edgar/data/109087...,https://www.sec.gov/Archives/edgar/data/109087...
2,A,24.0,2000-10-31,A,USD,1090872.0,2001-01-17,2001-01-17 00:00:00,2000.0,FY,...,0.108048,407000000.0,7.570000e+08,0.070268,1.68,1.66,449000000.0,455000000.0,https://www.sec.gov/Archives/edgar/data/109087...,https://www.sec.gov/Archives/edgar/data/109087...
3,A,23.0,2001-10-31,A,USD,1090872.0,2002-01-22,2002-01-22 00:00:00,2001.0,FY,...,-0.056813,-71000000.0,1.680000e+08,0.020010,0.38,0.38,458000000.0,458000000.0,https://www.sec.gov/Archives/edgar/data/109087...,https://www.sec.gov/Archives/edgar/data/109087...
4,A,22.0,2002-10-31,A,USD,1090872.0,2002-12-20,2002-12-20 17:27:53,2002.0,FY,...,-0.257404,-525000000.0,-1.032000e+09,-0.171714,-2.22,-2.22,465000000.0,465000000.0,https://www.sec.gov/Archives/edgar/data/109087...,https://www.sec.gov/Archives/edgar/data/109087...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153340,ZYXI,4.0,2019-12-31,ZYXI,USD,846475.0,2020-02-27,2020-02-27 17:24:43,2019.0,FY,...,0.262601,2449000.0,9.492000e+06,0.208744,0.27,0.25,35682898.0,37359298.0,https://www.sec.gov/Archives/edgar/data/846475...,https://www.sec.gov/Archives/edgar/data/846475...
153341,ZYXI,3.0,2020-12-31,ZYXI,USD,846475.0,2021-02-25,2021-02-25 17:01:35,2020.0,FY,...,0.126719,1079000.0,9.074000e+06,0.113252,0.24,0.24,37255898.0,38437298.0,https://www.sec.gov/Archives/edgar/data/846475...,https://www.sec.gov/Archives/edgar/data/846475...
153342,ZYXI,2.0,2021-12-31,ZYXI,USD,846475.0,2022-03-22,2022-03-21 19:19:49,2021.0,FY,...,0.170920,5168000.0,1.710300e+07,0.131258,0.41,0.40,42148698.0,43116698.0,https://www.sec.gov/Archives/edgar/data/846475...,https://www.sec.gov/Archives/edgar/data/846475...
153343,ZYXI,1.0,2022-12-31,ZYXI,USD,846475.0,2023-03-14,2023-03-13 18:49:43,2022.0,FY,...,0.140345,5150000.0,1.704800e+07,0.107785,0.44,0.44,38467000.0,39127000.0,https://www.sec.gov/Archives/edgar/data/846475...,https://www.sec.gov/Archives/edgar/data/846475...


## Normalizing USD as the Default Currency for Matching

In [23]:
# Farklı currencyler var ama aynı satırda hepsi aynı mı onu kontrol ediyoruz
currency_match = (
    (valid_cik_df["cf_reportedCurrency"] == valid_cik_df["is_reportedCurrency"]) &
    (valid_cik_df["cf_reportedCurrency"] == valid_cik_df["bs_reportedCurrency"])
)
matching_currency_df = valid_cik_df[currency_match]
mismatched_currency_df = valid_cik_df[~currency_match]
print(f"Aynı currency olan satır sayısı: {currency_match.sum()} / {len(valid_cik_df)}")

Aynı currency olan satır sayısı: 133038 / 133061


In [24]:
# Her üç kolondan tuple yaparak hangi kombinasyonların farklı olduğunu görelim
currency_combinations = (
    mismatched_currency_df[["cf_reportedCurrency", "is_reportedCurrency", "bs_reportedCurrency"]]
    .dropna()
    .value_counts()
    .sort_values(ascending=False)
)
print("En sık görülen farklı currency kombinasyonları:")
print(currency_combinations.head(10))

En sık görülen farklı currency kombinasyonları:
cf_reportedCurrency  is_reportedCurrency  bs_reportedCurrency
CAD                  USD                  CAD                    10
ILS                  USD                  ILS                     4
CAD                  CAD                  EUR                     2
                     EUR                  CAD                     2
                     USD                  USD                     1
HKD                  USD                  USD                     1
USD                  CAD                  CAD                     1
                                          USD                     1
                     CNY                  CNY                     1
Name: count, dtype: int64


In [25]:
# Hangi symbollerde var
conflict_symbols = mismatched_currency_df["symbol"].unique()
print(f"Currency uyuşmazlığı olan benzersiz sembol sayısı: {len(conflict_symbols)}")
print(conflict_symbols[:100])  # İlk 10 tanesini göster

Currency uyuşmazlığı olan benzersiz sembol sayısı: 14
['AACG' 'AMTD' 'BCTXW' 'BRAG' 'CP' 'ENBFF' 'EU' 'HITI' 'INM' 'ITRG'
 'JUSHF' 'RVSN' 'RVSNW' 'SII']


In [26]:
# Her üç sütunun da USD olup olmadığını kontrol eden maske ile sadece USD'leri tutuyorum
all_usd_mask = (
    (valid_cik_df["cf_reportedCurrency"] == "USD") &
    (valid_cik_df["is_reportedCurrency"] == "USD") &
    (valid_cik_df["bs_reportedCurrency"] == "USD")
)
valid_cik_df = valid_cik_df[all_usd_mask].copy()

In [27]:
# FMP ile symbolden research yapma toolu
"""
import requests

def read_api_key_from_file(file_path):
    with open(file_path, "r") as file:
        return file.read().strip()
def search_symbol_fmp(query, api_key, limit=10, exchange=None):
    base_url = "https://financialmodelingprep.com/api/v3/search"
    params = {
        "query": query,
        "limit": limit,
        "apikey": api_key
    }
    if exchange:
        params["exchange"] = exchange  
    response = requests.get(base_url, params=params)
    if response.status_code == 200:
        data = response.json()
        if data:
            print(f"🔍 {query} için bulunan {len(data)} sonuç:")
            for item in data:
                print(f"- Symbol: {item.get('symbol')}, Name: {item.get('name')}, Exchange: {item.get('stockExchange')}, Currency: {item.get('currency')}")
        else:
            print("❗ Sonuç bulunamadı.")
    else:
        print(f"❌ Hata: {response.status_code} - {response.text}")
api_key = read_api_key_from_file("FMP API KEY.txt")
search_symbol_fmp("AA", api_key)
"""

'\nimport requests\n\ndef read_api_key_from_file(file_path):\n    with open(file_path, "r") as file:\n        return file.read().strip()\ndef search_symbol_fmp(query, api_key, limit=10, exchange=None):\n    base_url = "https://financialmodelingprep.com/api/v3/search"\n    params = {\n        "query": query,\n        "limit": limit,\n        "apikey": api_key\n    }\n    if exchange:\n        params["exchange"] = exchange  \n    response = requests.get(base_url, params=params)\n    if response.status_code == 200:\n        data = response.json()\n        if data:\n            print(f"🔍 {query} için bulunan {len(data)} sonuç:")\n            for item in data:\n                print(f"- Symbol: {item.get(\'symbol\')}, Name: {item.get(\'name\')}, Exchange: {item.get(\'stockExchange\')}, Currency: {item.get(\'currency\')}")\n        else:\n            print("❗ Sonuç bulunamadı.")\n    else:\n        print(f"❌ Hata: {response.status_code} - {response.text}")\napi_key = read_api_key_from_file("

In [28]:
# Sadece NASDAQ içinde aramak istersen:
search_symbol_fmp("Tesla", api_key, limit=5, exchange="NASDAQ")

NameError: name 'search_symbol_fmp' is not defined

In [ ]:
# Datada farklı currencylerle yazıldığını inceliyoruz
bs_currencies = cleaned_df["bs_reportedCurrency"].dropna().unique().tolist()
cleaned_df["cf_reportedCurrency"].dropna().unique().tolist()

['USD',
 'CNY',
 'CAD',
 'BRL',
 'EUR',
 'CHF',
 'JPY',
 'CLP',
 'HKD',
 'MXN',
 'MYR',
 'GBP',
 'ZAR',
 'TWD',
 'AUD',
 'PEN',
 'COP',
 'ARS',
 'THB',
 'ILS',
 'TRY',
 'SEK',
 'DKK',
 'KRW',
 'INR',
 'SGD',
 'PHP',
 'KZT',
 'NZD',
 'IDR',
 'NOK',
 'VND']

In [ ]:
# Farklı currency'leri text olarak kaydediyoruz
bs_currencies = cleaned_df["bs_reportedCurrency"].dropna().unique().tolist()
cf_currencies = cleaned_df["cf_reportedCurrency"].dropna().unique().tolist()
all_currencies = sorted(set(bs_currencies + cf_currencies))
with open("different_currencies.txt", "w") as f:
    for currency in all_currencies:
        f.write(currency + "\n")

## Removing Rows with Inconsistent Reporting Dates

In [ ]:
date_match = (
    (valid_cik_df["cf_date"] == valid_cik_df["is_date"]) &
    (valid_cik_df["cf_date"] == valid_cik_df["bs_date"])
)
matching_date_df = valid_cik_df[date_match]
mismatched_date_df = valid_cik_df[~date_match]
mismatched_date_df[["bs_date","is_date","cf_date"]]

,bs_date,is_date,cf_date
2135,2011-06-30,2012-06-30,2011-06-30
2136,2012-06-30,2012-12-31,2012-06-30
2211,2024-03-31,2023-12-31,2023-12-31
7529,2016-12-31,2016-09-30,2016-09-30
10330,2022-09-30,2023-03-31,2022-09-30
...,...,...,...
150943,2000-12-31,2000-04-29,2000-04-29
151198,2015-06-30,2014-12-31,2014-12-31
151201,2018-03-31,2017-12-31,2017-12-31
153121,2015-12-31,2015-12-31,2016-02-29


In [ ]:
len(valid_cik_df)

123851

In [ ]:
# Tarihler arasında tam eşleşme kontrolü ve eşleşenleri tutma
date_match = (
    (valid_cik_df["cf_date"] == valid_cik_df["is_date"]) &
    (valid_cik_df["cf_date"] == valid_cik_df["bs_date"])
)
valid_cik_df = valid_cik_df[date_match].copy()
len(valid_cik_df)

123618

In [ ]:
valid_cik_df["bs_cik"].nunique()

5790

In [ ]:
cleaned_df["bs_cik"].nunique()

6225

# Integrating Financial Metrics

In [30]:
# Sütun isimlerini texte kaydet
column_names = valid_cik_df.columns.tolist()
with open("valid_cik_df_columns.txt", "w") as f:
    for col in column_names:
        f.write(col + "\n")

In [31]:
# Finansal metrikler ekleyip hataları kontrol ediyoruz
def add_financial_ratios(df):
    ratio_definitions = {
        "gross_ROA": lambda df: (df["is_revenue"] - df["is_costOfRevenue"]) / df["bs_totalCurrentAssets"],
        "EBIT_to_MP": lambda df: df["is_ebitda"] / df["bs_totalStockholdersEquity"],
        "EBIT_to_Equity": lambda df: df["is_ebitda"] / df["bs_totalStockholdersEquity"],
        "netIncome_margin": lambda df: df["is_netIncome"] / df["is_revenue"],
        "ROE": lambda df: df["is_netIncome"] / df["bs_totalStockholdersEquity"],
        "ROA": lambda df: df["is_netIncome"] / df["bs_totalAssets"],
        "debt_to_equity": lambda df: df["bs_totalLiabilities"] / df["bs_totalStockholdersEquity"],
        "current_ratio": lambda df: df["bs_totalCurrentAssets"] / df["bs_totalCurrentLiabilities"],
        "quick_ratio": lambda df: (df["bs_cashAndShortTermInvestments"] + df["bs_netReceivables"]) / df["bs_totalCurrentLiabilities"],
        "interest_coverage": lambda df: df["is_ebitda"] / df["is_interestExpense"],
        "asset_turnover": lambda df: df["is_revenue"] / df["bs_totalAssets"],
        "gross_margin": lambda df: (df["is_revenue"] - df["is_costOfRevenue"]) / df["is_revenue"],
        "net_margin": lambda df: df["is_netIncome"] / df["is_revenue"],
        "operating_margin": lambda df: df["is_operatingIncome"] / df["is_revenue"],
        "ebitda_margin": lambda df: df["is_ebitda"] / df["is_revenue"],
        "receivables_turnover": lambda df: df["is_revenue"] / df["bs_netReceivables"],
        "inventory_turnover": lambda df: df["is_costOfRevenue"] / df["bs_inventory"],
        "debt_to_assets": lambda df: df["bs_totalLiabilities"] / df["bs_totalAssets"],
        "longterm_debt_to_equity": lambda df: df["bs_longTermDebt"] / df["bs_totalStockholdersEquity"],
        "operating_cf_to_netincome": lambda df: df["cf_netCashProvidedByOperatingActivities"] / df["is_netIncome"],
        "free_cf_to_assets": lambda df: df["cf_freeCashFlow"] / df["bs_totalAssets"],
        "free_cf_to_equity": lambda df: df["cf_freeCashFlow"] / df["bs_totalStockholdersEquity"],
        "rz_external_finance": lambda df: (df["cf_commonStockIssued"] - df["cf_commonStockRepurchased"] + df["cf_debtRepayment"]) / df["bs_totalAssets"],
        "tobins_q": lambda df: df["bs_totalAssets"] / df["bs_totalStockholdersEquity"]
    }
    inverse_metrics = {
        "debt_to_equity",
        "debt_to_assets",
        "longterm_debt_to_equity"
    }
    error_log = {}
    for name, func in ratio_definitions.items():
        try:
            df[name] = func(df)
        except Exception as e:
            error_log[name] = str(e)
            print(f"Error calculating {name}: {e}")

    return df, error_log, inverse_metrics
valid_cik_df, ratio_errors, inverse_metrics = add_financial_ratios(valid_cik_df)
ratio_errors

{}

In [32]:
# Finansal oranlar tanımındaki isimleri kullanarak sonsuz değerleri temizliyoruz
for col in valid_cik_df.columns:
    if col in ratio_errors:
        continue
    if col in [
        "gross_ROA", "EBIT_to_MP", "EBIT_to_Equity", "netIncome_margin", "ROE", "ROA",
        "debt_to_equity", "current_ratio", "quick_ratio", "interest_coverage", "asset_turnover",
        "gross_margin", "net_margin", "operating_margin", "ebitda_margin", "receivables_turnover",
        "inventory_turnover", "debt_to_assets", "longterm_debt_to_equity",
        "operating_cf_to_netincome", "free_cf_to_assets", "free_cf_to_equity",
        "rz_external_finance", "tobins_q"
    ]:
        valid_cik_df[col] = valid_cik_df[col].replace([np.inf, -np.inf], np.nan)

In [33]:
# Ranking yapmak istiyoruz
ranking_metrics = [
    "gross_ROA", "EBIT_to_MP", "ROE", "ROA", "netIncome_margin",
    "debt_to_equity", "current_ratio", "quick_ratio", "asset_turnover"
]
inverse_metrics = {"debt_to_equity"}
ranked_results = []
for idx, row in df4.iterrows():
    year = row['year']
    prev_year = year - 1
    tickers = row[1:].dropna().unique().tolist()
    df_year = valid_cik_df[
        (valid_cik_df['calendarYear'] == prev_year) &
        (valid_cik_df['symbol'].isin(tickers))
    ].copy()
    if df_year.empty:
        continue
    for metric in ranking_metrics:
        if metric in df_year.columns:
            ascending = metric in inverse_metrics  
            df_year[f"rank_{metric}"] = df_year[metric].rank(ascending=ascending, method='min', na_option='bottom')
    rank_cols = [f"rank_{m}" for m in ranking_metrics if f"rank_{m}" in df_year.columns]
    df_year["combined_score"] = df_year[rank_cols].sum(axis=1)
    df_year["final_rank"] = df_year["combined_score"].rank(method="min")
    df_year["ranking_year"] = year
    ranked_results.append(df_year)
ranked_df = pd.concat(ranked_results, ignore_index=True)
ranked_df = ranked_df.sort_values(by=["ranking_year", "final_rank"]).reset_index(drop=True)

In [35]:
ranked_df

,bs_Symbol,bs_Unnamed: 1,bs_date,bs_symbol,bs_reportedCurrency,bs_cik,bs_fillingDate,bs_acceptedDate,bs_calendarYear,bs_period,...,rank_ROA,rank_netIncome_margin,rank_debt_to_equity,rank_current_ratio,rank_quick_ratio,rank_asset_turnover,combined_score,final_rank,ranking_year,year
0,ADBE,25.0,1999-12-03,ADBE,USD,796343.0,2000-02-16,2000-02-16 00:00:00,1999.0,FY,...,1.0,21.0,27.0,30.0,15.0,53.0,273.0,1.0,2000,1999
1,AMGN,25.0,1999-12-31,AMGN,USD,318154.0,2000-03-07,2000-03-07 00:00:00,1999.0,FY,...,2.0,8.0,12.0,24.0,17.0,136.0,380.0,2.0,2000,1999
2,BMY,25.0,1999-12-31,BMY,USD,14272.0,2000-03-30,2000-03-30 00:00:00,1999.0,FY,...,3.0,28.0,61.0,70.0,55.0,64.0,389.0,3.0,2000,1999
3,RHI,25.0,1999-12-31,RHI,USD,315213.0,2000-03-08,2000-03-08 00:00:00,1999.0,FY,...,10.0,140.0,13.0,17.0,5.0,11.0,390.0,4.0,2000,1999
4,ORCL,25.0,1999-05-31,ORCL,USD,1341439.0,1999-05-31,1999-05-30 20:00:00,1999.0,FY,...,11.0,49.0,58.0,54.0,26.0,58.0,453.0,5.0,2000,1999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9770,MAA,0.0,2024-12-31,MAA,USD,912595.0,2025-02-07,2025-02-07 16:16:04,2024.0,FY,...,205.0,329.0,107.0,309.0,309.0,323.0,2340.0,326.0,2025,2024
9771,BAC,0.0,2024-12-31,BAC,USD,70858.0,2024-12-31,2024-12-30 19:00:00,2024.0,FY,...,304.0,40.0,305.0,296.0,260.0,319.0,2349.0,327.0,2025,2024
9772,GS,0.0,2024-12-31,GS,USD,886982.0,2024-12-31,2024-12-31 00:00:00,2024.0,FY,...,303.0,36.0,311.0,317.0,317.0,320.0,2358.0,328.0,2025,2024
9773,ALB,0.0,2024-12-31,ALB,USD,915913.0,2025-02-12,2025-02-12 17:09:12,2024.0,FY,...,320.0,326.0,71.0,311.0,314.0,258.0,2414.0,329.0,2025,2024


In [37]:
ranked_df["year"] = pd.to_datetime(ranked_df["bs_date"]).dt.year
top5_each_year = ranked_df.groupby("year", group_keys=False).apply(lambda x: x.nsmallest(5, "final_rank"))
top5_each_year[top5_each_year["year"] == 2002][["bs_symbol","ranking_year","combined_score"]]

/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_1116/3291675203.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top5_each_year = ranked_df.groupby("year", group_keys=False).apply(lambda x: x.nsmallest(5, "final_rank"))


,bs_symbol,ranking_year,combined_score
797,TROW,2003,361.0
798,ADBE,2003,418.0
799,ORCL,2003,465.0
800,ZBH,2003,538.0
801,JNJ,2003,554.0


# Applying Financial Ratios for Performance-Based Ranking

In [ ]:
#!/usr/bin/env python
import time, json, certifi
import pandas as pd
import numpy  as np
from datetime import datetime
try:
    from urllib.request import urlopen
except ImportError:           # Py-2 fallback
    from urllib2 import urlopen

# -------------------------------------------------- #
API_KEY   = open("FMP API KEY.txt").read().strip()
BASE_URL  = "https://financialmodelingprep.com/api/v3/"
# -------------------------------------------------- #

def get_jsonparsed_data(url: str) -> dict | list:
    """URL → JSON → Python object (urllib, SSL-cert handled)."""
    with urlopen(url, cafile=certifi.where()) as resp:
        return json.loads(resp.read().decode("utf-8"))

def fmp_close_series(ticker: str, start: str, end: str) -> pd.Series | None:
    """Tek sembol, tarih aralığı kapaniş fiyat serisi döndürür."""
    url = (f"{BASE_URL}historical-price-full/{ticker}"
           f"?from={start}&to={end}&apikey={API_KEY}")
    js  = get_jsonparsed_data(url)
    if not js or "historical" not in js or not js["historical"]:
        return None
    df = pd.DataFrame(js["historical"])
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date").sort_index()
    return df["close"]

portfolio_returns = []

for year in range(2000, 2025):
    buy_date  = f"{year}-04-01"
    sell_date = f"{year+1}-03-30"

    # ── o yılın ilk 20 hissesi
    tickers = (
        ranked_df[ ranked_df["ranking_year"] == year ]
        .nsmallest(20, "final_rank")["symbol"]
        .dropna().unique().tolist()
    )
    if not tickers:
        continue

    returns = []

    for tkr in tickers:
        s = fmp_close_series(tkr, buy_date, sell_date)
        if s is None or s.empty:
            continue

        # en yakın borsa günlerini bul
        idx_buy  = s.index.get_indexer([pd.Timestamp(buy_date )], method="nearest")[0]
        idx_sell = s.index.get_indexer([pd.Timestamp(sell_date)], method="nearest")[0]
        if idx_buy == -1 or idx_sell == -1:
            continue

        buy_px  = s.iloc[idx_buy]
        sell_px = s.iloc[idx_sell]
        returns.append((sell_px / buy_px) - 1)

        time.sleep(0.25)        

    if returns:
        portfolio_returns.append({
            "year"          : year,
            "average_return": np.mean(returns),
            "top20_tickers" : ", ".join(tickers)
        })

    time.sleep(1)              

# ── Sonuç tablosu
results_df = pd.DataFrame(portfolio_returns)
print(results_df.head())
results_df.to_csv("simulated_portfolio_returns_FMP.csv", index=False)


/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_959/97898391.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:
/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_959/97898391.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:
/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_959/97898391.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:
/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_959/97898391.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:
/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_959/97898391.py:18: DeprecationWa

   year  average_return                                      top20_tickers
0  2000       -0.043735  ADBE, AMGN, BMY, RHI, ORCL, LLY, ADSK, JNJ, IN...
1  2001        0.150891  ADBE, ZBH, TROW, BMY, RHI, WAT, MDT, NVDA, AMG...
2  2002       -0.165474  ADBE, ORCL, TROW, JNJ, PFE, MDT, CTAS, AMGN, A...
3  2003        0.340114  TROW, ADBE, ORCL, ZBH, JNJ, HSY, HOG, UPS, PFE...
4  2004        0.136991  TPR, ORCL, TROW, ADBE, JNJ, MDT, PGR, HSY, WAT...


In [155]:
results_df

,year,average_return,top20_tickers
0,2000,-0.043735,"ADBE, AMGN, BMY, RHI, ORCL, LLY, ADSK, JNJ, IN..."
1,2001,0.150891,"ADBE, ZBH, TROW, BMY, RHI, WAT, MDT, NVDA, AMG..."
2,2002,-0.165474,"ADBE, ORCL, TROW, JNJ, PFE, MDT, CTAS, AMGN, A..."
3,2003,0.340114,"TROW, ADBE, ORCL, ZBH, JNJ, HSY, HOG, UPS, PFE..."
4,2004,0.136991,"TPR, ORCL, TROW, ADBE, JNJ, MDT, PGR, HSY, WAT..."
5,2005,0.171325,"TPR, TROW, ADBE, ORCL, NUE, JNJ, PGR, GILD, HO..."
6,2006,0.138383,"TPR, TROW, ADBE, ANF, RHI, NUE, HOG, MCO, JNJ,..."
7,2007,-0.073999,"TPR, FCX, TXN, TROW, ICE, NUE, ANF, RHI, ADSK,..."
8,2008,-0.426145,"TPR, TROW, GILD, TXN, ANF, FAST, RHI, HAL, NE,..."
9,2009,0.603526,"TPR, FAST, GILD, TROW, WAT, RHI, XOM, NE, CF, ..."


In [ ]:
overall_average = results_df["average_return"].mean()
print(f"Portföyün 2000-2024 dönemi ortalama yıllık getirisi: {overall_average:.4%}")

Portföyün 2000-2024 dönemi ortalama yıllık getirisi: 13.6982%


In [39]:
ranked_df["year"] = pd.to_datetime(ranked_df["bs_date"]).dt.year
top5_each_year = ranked_df.groupby("year", group_keys=False).apply(lambda x: x.nsmallest(5, "final_rank"))
top5_each_year[top5_each_year["year"]==2024]

/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_1116/647830622.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top5_each_year = ranked_df.groupby("year", group_keys=False).apply(lambda x: x.nsmallest(5, "final_rank"))


,bs_Symbol,bs_Unnamed: 1,bs_date,bs_symbol,bs_reportedCurrency,bs_cik,bs_fillingDate,bs_acceptedDate,bs_calendarYear,bs_period,...,rank_ROA,rank_netIncome_margin,rank_debt_to_equity,rank_current_ratio,rank_quick_ratio,rank_asset_turnover,combined_score,final_rank,ranking_year,year
9445,HON,0.0,2024-12-31,HON,USD,773840.0,2025-02-14,2025-02-14 14:50:25,2024.0,FY,...,3.0,134.0,48.0,56.0,3.0,3.0,262.0,1.0,2025,2024
9446,NVDA,0.0,2024-01-28,NVDA,USD,1045810.0,2024-02-21,2024-02-21 16:36:57,2024.0,FY,...,6.0,4.0,60.0,24.0,21.0,73.0,426.0,2.0,2025,2024
9447,TPL,0.0,2024-12-31,TPL,USD,1811074.0,2025-02-19,2025-02-19 16:21:43,2024.0,FY,...,8.0,1.0,29.0,8.0,7.0,160.0,447.0,3.0,2025,2024
8941,LULU,0.0,2024-01-28,LULU,USD,1397187.0,2024-03-21,2024-03-21 16:12:29,2023.0,FY,...,20.0,182.0,98.0,76.0,108.0,50.0,825.0,4.0,2024,2024
9448,META,0.0,2024-12-31,META,USD,1326801.0,2025-01-30,2025-01-29 20:00:50,2024.0,FY,...,21.0,10.0,55.0,40.0,28.0,153.0,589.0,4.0,2025,2024


# Değişik Denemeler

In [ ]:
# Farklı bir ranking deniyoruz
ranking_metrics = [
    "gross_ROA", "EBIT_to_MP", "ROE", "ROA", "netIncome_margin",
    "debt_to_equity", "current_ratio", "quick_ratio", "asset_turnover"
]
inverse_metrics = {"debt_to_equity"}
ranked_results = []
for idx, row in df4.iterrows():
    year = row['year']
    prev_year = year - 1
    tickers = row[1:].dropna().unique().tolist()
    df_year = valid_cik_df[
        (valid_cik_df['calendarYear'] == prev_year) &
        (valid_cik_df['symbol'].isin(tickers))
    ].copy()
    if df_year.empty:
        continue
    for metric in ranking_metrics:
        if metric in df_year.columns:
            ascending = metric in inverse_metrics  
            df_year[f"rank_{metric}"] = df_year[metric].rank(ascending=ascending, method='min', na_option='bottom')
    rank_cols = [f"rank_{m}" for m in ranking_metrics if f"rank_{m}" in df_year.columns]
    df_year["combined_score"] = df_year[rank_cols].sum(axis=1)
    df_year["final_rank"] = df_year["combined_score"].rank(method="min")
    df_year["ranking_year"] = year
    ranked_results.append(df_year)
ranked_df = pd.concat(ranked_results, ignore_index=True)
ranked_df = ranked_df.sort_values(by=["ranking_year", "final_rank"]).reset_index(drop=True)

# Senator Deneme

In [ ]:
# Senator verisi çekme
import requests
import pandas as pd
from datetime import datetime, timedelta

symbol = "AAPL"
API_KEY = open("FMP API KEY.txt").read().strip()
url = f"https://financialmodelingprep.com/api/v4/senate-trading?symbol={symbol}&apikey={API_KEY}"

response = requests.get(url)
data = response.json()

df = pd.DataFrame(data)
df['transactionDate'] = pd.to_datetime(df['transactionDate'])
df.sort_values(by='transactionDate', ascending=False, inplace=True)
df

,firstName,lastName,office,link,dateRecieved,transactionDate,owner,assetDescription,assetType,type,amount,comment,symbol
0,Tommy,Tuberville,Tommy Tuberville,https://efdsearch.senate.gov/search/view/ptr/6...,2025-05-15,2025-04-15,Joint,Apple Inc,Stock,Sale,"$15,001 - $50,000",,AAPL
1,Tommy,Tuberville,Tommy Tuberville,https://efdsearch.senate.gov/search/view/ptr/6...,2025-05-14,2025-04-15,Joint,Apple Inc,Stock,Sale (Full),"$15,001 - $50,000",--,AAPL
2,Sheldon,Whitehouse,Sheldon Whitehouse,https://efdsearch.senate.gov/search/view/ptr/3...,2025-05-13,2025-04-14,Self,Apple Inc,Stock,Sale,"$15,001 - $50,000",,AAPL
3,Sheldon,Whitehouse,Sheldon Whitehouse,https://efdsearch.senate.gov/search/view/ptr/3...,2025-05-12,2025-04-14,Self,Apple Inc,Stock,Sale (Partial),"$15,001 - $50,000",--,AAPL
4,Shelley,Moore Capito,Shelley Moore Capito,https://efdsearch.senate.gov/search/view/ptr/2...,2025-03-05,2025-02-05,Spouse,Apple Inc,Stock,Sale (Partial),"$1,001 - $15,000",--,AAPL


In [5]:
df.columns

Index(['firstName', 'lastName', 'office', 'link', 'dateRecieved',
       'transactionDate', 'owner', 'assetDescription', 'assetType', 'type',
       'amount', 'comment', 'symbol'],
      dtype='object')

In [ ]:
# Yapılacaklar
# Eski sistemdeki deterministik metrikler ekonomik
# Top 20yi seçmiycek bir puanlama üstünü alacak oturmuş bir puanlama
# Senatör checki yapılcak satması takip edilecek yıl bazlı işlem değil satış tarihi değişebilecek
# Senatör checki alımlarda da pozitif etkenli bir faz olucak ranking sonrası
# Meclis temsilcilerini de ekle
# Etl fazı da ekle
# Fraud detection negatif değişkenli bir faz olacak
# Parametrizasyon güncellemesi için YK şeyleri kullanılacak


In [23]:
import time, json, certifi
import pandas as pd
import numpy  as np
from datetime import datetime, timedelta

try:
    from urllib.request import urlopen
except ImportError:
    from urllib2 import urlopen

# -------------------------------------------------- #
API_KEY   = open("FMP API KEY.txt").read().strip()
BASE_URL  = "https://financialmodelingprep.com/api/v3/"
# -------------------------------------------------- #

def get_jsonparsed_data(url: str):
    """URL'den JSON alıp Python objesine çevirir (SSL sertifikası ile)."""
    with urlopen(url, cafile=certifi.where()) as resp:
        return json.loads(resp.read().decode("utf-8"))

def fmp_close_series(ticker: str, start: str, end: str) -> pd.Series:
    """
    Tek bir hisse için belirli tarih aralığındaki kapanış fiyat serisini döndürür.
    """
    url = (f"{BASE_URL}historical-price-full/{ticker}"
           f"?from={start}&to={end}&apikey={API_KEY}")
    js = get_jsonparsed_data(url)
    if not js or "historical" not in js or not js["historical"]:
        return pd.Series()
    df = pd.DataFrame(js["historical"])
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date").sort_index()
    return df["close"]

# -------------------------------------------
# 1. Fonksiyon: Son 1 Yıldaki Senatör İşlemleriyle Hisse Puanlaması
def score_stock_by_senator_trades(ticker: str) -> None:
    """
    Verilen hisse (ticker) için son 1 yıldaki senatör alım işlemlerini inceleyerek,
    her işlemde alım gününün etrafındaki fiyat aralığı içinde (±3 gün) alım fiyatının
    göreceli konumunu hesaplar. Düşük değerler (dip yakınında alım) daha iyi kabul edilir.
    Ortalama "relative score" üzerinden, 0-100 arasında bir puan hesaplanır.
    """
    # Senatör işlemlerini çek (alım tipinde) – burada tüm veriler çekiliyor.
    url = f"https://financialmodelingprep.com/api/v4/senate-trading?symbol={ticker}&apikey={API_KEY}"
    data = get_jsonparsed_data(url)
    trades_df = pd.DataFrame(data)
    if trades_df.empty:
        print(f"No senator trades found for {ticker}.")
        return

    trades_df['transactionDate'] = pd.to_datetime(trades_df['transactionDate'])
    # Filtre: sadece "purchase" veya "buy" tipinde olanlar (küçük-büyük harf duyarsız)
    trades_df = trades_df[trades_df['type'].str.lower().isin(['purchase', 'buy'])]
    # Tarih filtresi: Son 1 yıl
    one_year_ago = datetime.now() - timedelta(days=365)
    trades_df = trades_df[trades_df['transactionDate'] >= one_year_ago]
    if trades_df.empty:
        print(f"No senator trades in the last one year for {ticker}.")
        return

    # Senatör ismini oluştur
    trades_df['senator'] = trades_df['firstName'] + ' ' + trades_df['lastName']

    scores = []
    for _, row in trades_df.iterrows():
        t_date = row['transactionDate']
        # ±3 gün aralığı belirle:
        start = (t_date - timedelta(days=3)).strftime('%Y-%m-%d')
        end   = (t_date + timedelta(days=3)).strftime('%Y-%m-%d')
        price_series = fmp_close_series(ticker, start, end)
        if price_series.empty:
            continue

        # İşlem gününe en yakın kapanış fiyatını bul:
        idx = price_series.index.get_indexer([t_date], method="nearest")[0]
        trade_price = price_series.iloc[idx]

        # Aralık içindeki min ve max fiyatlar:
        min_px = price_series.min()
        max_px = price_series.max()

        # Fiyat aralığındaki göreceli konum hesaplanır.
        # 0: En düşük fiyattan, 1: En yüksek fiyattan
        relative_position = (trade_price - min_px) / (max_px - min_px + 1e-6)
        # Daha düşük relative_position iyi kabul edilir; bunu tersine çeviriyoruz.
        score = (1 - relative_position) * 100
        scores.append(score)

    if scores:
        average_score = np.mean(scores)
        print(f"{ticker} için senatör alım işlemleri ortalama puanı: {average_score:.2f} (0=düşük, 100=çok düşük fiyattan alım)")
    else:
        print("Hiç işlem verisine ulaşılamadı.")

# -------------------------------------------
# 2. Fonksiyon: Verilen Tarih Aralığında, Senatörlerin 'dateRecieved' Tarihli Rapor Fiyatı ile Karşılaştırma
def compare_trade_vs_received_price(ticker: str, start_range: str, end_range: str) -> pd.DataFrame:
    url = f"https://financialmodelingprep.com/api/v4/senate-trading?symbol={ticker}&apikey={API_KEY}"
    data = get_jsonparsed_data(url)
    trades_df = pd.DataFrame(data)
    if trades_df.empty:
        print(f"No senator trades found for {ticker}.")
        return pd.DataFrame()

    trades_df['transactionDate'] = pd.to_datetime(trades_df['transactionDate'])
    trades_df['dateRecieved'] = pd.to_datetime(trades_df['dateRecieved'])
    trades_df = trades_df[trades_df['type'].str.lower().isin(['purchase', 'buy'])]

    mask = (trades_df['dateRecieved'] >= pd.to_datetime(start_range)) & (trades_df['dateRecieved'] <= pd.to_datetime(end_range))
    trades_df = trades_df.loc[mask]
    if trades_df.empty:
        print(f"No senator trades with dateRecieved between {start_range} and {end_range} for {ticker}.")
        return pd.DataFrame()

    trades_df['senator'] = trades_df['firstName'] + ' ' + trades_df['lastName']

    result_rows = []

    for _, row in trades_df.iterrows():
        trans_date = row['transactionDate']
        received_date = row['dateRecieved']
        senator = row['senator']

        start_td = (trans_date - timedelta(days=3)).strftime('%Y-%m-%d')
        end_td   = (trans_date + timedelta(days=3)).strftime('%Y-%m-%d')
        trans_price_series = fmp_close_series(ticker, start_td, end_td)
        if trans_price_series.empty:
            continue
        idx_trans = trans_price_series.index.get_indexer([trans_date], method="nearest")[0]
        transaction_price = trans_price_series.iloc[idx_trans]

        start_rd = (received_date - timedelta(days=3)).strftime('%Y-%m-%d')
        end_rd   = (received_date + timedelta(days=3)).strftime('%Y-%m-%d')
        received_price_series = fmp_close_series(ticker, start_rd, end_rd)
        if received_price_series.empty:
            continue
        idx_rec = received_price_series.index.get_indexer([received_date], method="nearest")[0]
        received_price = received_price_series.iloc[idx_rec]

        diff = received_price - transaction_price
        diff_pct = (diff / transaction_price) * 100

        result_rows.append({
            "Senator": senator,
            "Transaction Date": trans_date.date(),
            "Received Date": received_date.date(),
            "Transaction Price": round(transaction_price, 2),
            "Received Price": round(received_price, 2),
            "Price Diff ($)": round(diff, 2),
            "Price Diff (%)": round(diff_pct, 2),
        })

        time.sleep(0.25)

    result_df = pd.DataFrame(result_rows)
    return result_df

# -------------------------------------------
# Örnek Kullanımlar:
if __name__ == "__main__":
    # 1. Son 1 yılın senatör alım işlemleri üzerinden hisse puanlaması:
    print("=== Senatör Trade Score ===")
    score_stock_by_senator_trades("AAPL")  # örneğin AAPL için puanlama

    # 2. Belirlenen tarih aralığında transactionPrice ile receivedPrice karşılaştırması:
    print("\n=== Transaction vs Received Price Comparison ===")
    # Örnek: 2023-01-01 ile 2023-12-31 tarih aralığında
    compare_trade_vs_received_price("AAPL", "2023-01-01", "2023-12-31")


=== Senatör Trade Score ===


/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_13890/1544528688.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:


No senator trades in the last one year for AAPL.

=== Transaction vs Received Price Comparison ===


/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_13890/1544528688.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:
/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_13890/1544528688.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:
/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_13890/1544528688.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:
/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_13890/1544528688.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:
/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_13890/1544528688.

In [26]:
# 2. Fonksiyon: Verilen Tarih Aralığında, Senatörlerin 'dateRecieved' Tarihli Rapor Fiyatı ile Karşılaştırma
def compare_trade_vs_received_price(ticker: str, start_range: str, end_range: str) -> None:
    """
    Verilen hisse için; belirlenen tarih aralığında (start_range, end_range)
    senatörlerin raporlanan (dateRecieved) işlemlerini bulur ve:
      - Alımın yapıldığı transactionDate günündeki fiyatı
      - Raporun (dateRecieved) tarihindeki kapanış fiyatını
    getirerek, ikisi arasındaki farkı senatör ismiyle birlikte ekrana yazdırır.
    """
    url = f"https://financialmodelingprep.com/api/v4/senate-trading?symbol={ticker}&apikey={API_KEY}"
    data = get_jsonparsed_data(url)
    trades_df = pd.DataFrame(data)
    if trades_df.empty:
        print(f"No senator trades found for {ticker}.")
        return

    trades_df['transactionDate'] = pd.to_datetime(trades_df['transactionDate'])
    trades_df['dateRecieved'] = pd.to_datetime(trades_df['dateRecieved'])
    # Sadece alım işlemleri
    trades_df = trades_df[trades_df['type'].str.lower().isin(['purchase', 'buy'])]
    # Tarih aralığı filtresi: dateRecieved alanına göre
    mask = (trades_df['dateRecieved'] >= pd.to_datetime(start_range)) & (trades_df['dateRecieved'] <= pd.to_datetime(end_range))
    trades_df = trades_df.loc[mask]
    if trades_df.empty:
        print(f"No senator trades with dateRecieved between {start_range} and {end_range} for {ticker}.")
        return

    trades_df['senator'] = trades_df['firstName'] + ' ' + trades_df['lastName']

    for _, row in trades_df.iterrows():
        trans_date = row['transactionDate']
        received_date = row['dateRecieved']
        senator = row['senator']

        # Transaction price: transactionDate +/- 3 gün penceresinden al
        start_td = (trans_date - timedelta(days=3)).strftime('%Y-%m-%d')
        end_td   = (trans_date + timedelta(days=3)).strftime('%Y-%m-%d')
        trans_price_series = fmp_close_series(ticker, start_td, end_td)
        if trans_price_series.empty:
            continue
        idx_trans = trans_price_series.index.get_indexer([trans_date], method="nearest")[0]
        transaction_price = trans_price_series.iloc[idx_trans]

        # Received price: dateRecieved etrafında (±3 gün) fiyatı al
        start_rd = (received_date - timedelta(days=3)).strftime('%Y-%m-%d')
        end_rd   = (received_date + timedelta(days=3)).strftime('%Y-%m-%d')
        received_price_series = fmp_close_series(ticker, start_rd, end_rd)
        if received_price_series.empty:
            continue
        idx_rec = received_price_series.index.get_indexer([received_date], method="nearest")[0]
        received_price = received_price_series.iloc[idx_rec]

        diff = received_price - transaction_price
        diff_pct = (diff / transaction_price) * 100

        print(f"{senator} | Transaction Date: {trans_date.date()} | Transaction Price: ${transaction_price:.2f}")
        print(f"    Received Date: {received_date.date()} | Received Price: ${received_price:.2f}")
        print(f"    Fark: ${diff:.2f} ({diff_pct:.2f}%)\n")

        time.sleep(0.25)

if __name__ == "__main__":
    # 1. Son 1 yılın senatör alım işlemleri üzerinden hisse puanlaması:
    print("=== Senatör Trade Score ===")
    score_stock_by_senator_trades("AAPL")  # örneğin AAPL için puanlama

    # 2. Belirlenen tarih aralığında transactionPrice ile receivedPrice karşılaştırması:
    print("\n=== Transaction vs Received Price Comparison ===")
    # Örnek: 2023-01-01 ile 2023-12-31 tarih aralığında
    compare_trade_vs_received_price("AAPL", "2023-01-01", "2023-12-31")

=== Senatör Trade Score ===


/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_13890/1544528688.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:


No senator trades in the last one year for AAPL.

=== Transaction vs Received Price Comparison ===


/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_13890/1544528688.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:
/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_13890/1544528688.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:
/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_13890/1544528688.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:


Markwayne Mullin | Transaction Date: 2023-10-03 | Transaction Price: $172.40
    Received Date: 2023-10-31 | Received Price: $170.77
    Fark: $-1.63 (-0.95%)



/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_13890/1544528688.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:
/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_13890/1544528688.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:


Markwayne Mullin | Transaction Date: 2023-09-13 | Transaction Price: $174.21
    Received Date: 2023-10-11 | Received Price: $179.80
    Fark: $5.59 (3.21%)



/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_13890/1544528688.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:
/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_13890/1544528688.py:18: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  with urlopen(url, cafile=certifi.where()) as resp:


Shelley Moore Capito | Transaction Date: 2023-06-14 | Transaction Price: $183.95
    Received Date: 2023-07-16 | Received Price: $193.99
    Fark: $10.04 (5.46%)



In [29]:
#!/usr/bin/env python
import time, json, certifi
import pandas as pd
import numpy  as np
from datetime import datetime, timedelta
import ssl
from urllib.request import urlopen

# -------------------------------------------------- #
API_KEY   = open("FMP API KEY.txt").read().strip()
BASE_URL  = "https://financialmodelingprep.com/api/v3/"
# -------------------------------------------------- #

# 🔐 SSL uyumlu güncel veri çekme fonksiyonu
def get_jsonparsed_data(url: str):
    """URL → JSON → Python objesi (modern SSL context ile)."""
    context = ssl.create_default_context(cafile=certifi.where())
    with urlopen(url, context=context) as resp:
        return json.loads(resp.read().decode("utf-8"))

# 📈 Hisse kapanış fiyatlarını getir
def fmp_close_series(ticker: str, start: str, end: str) -> pd.Series:
    """
    Verilen hisse (ticker) ve tarih aralığı için kapanış fiyat serisini döndürür.
    """
    url = f"{BASE_URL}historical-price-full/{ticker}?from={start}&to={end}&apikey={API_KEY}"
    js = get_jsonparsed_data(url)
    if not js or "historical" not in js or not js["historical"]:
        return pd.Series()
    df = pd.DataFrame(js["historical"])
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date").sort_index()
    return df["close"]

# 🧮 Ana analiz fonksiyonu: transactionDate vs. dateRecieved
def compare_trade_vs_received_price(ticker: str, start_range: str, end_range: str) -> pd.DataFrame:
    """
    Senatörlerin bir hisseyi belirli bir tarih aralığında (transactionDate) aldıkları
    günkü fiyat ile raporun (dateRecieved) bildirildiği günkü fiyatı karşılaştırır.
    """
    url = f"https://financialmodelingprep.com/api/v4/senate-trading?symbol={ticker}&apikey={API_KEY}"
    data = get_jsonparsed_data(url)
    trades_df = pd.DataFrame(data)

    if trades_df.empty:
        print(f"No senator trades found for {ticker}.")
        return pd.DataFrame()

    trades_df['transactionDate'] = pd.to_datetime(trades_df['transactionDate'])
    trades_df['dateRecieved'] = pd.to_datetime(trades_df['dateRecieved'])

    # Sadece 'purchase' türü işlemleri al
    trades_df = trades_df[trades_df['type'].str.lower().isin(['purchase', 'buy'])]

    # 🔍 Filtreleme: transactionDate, kullanıcı verdiği aralıkta olmalı
    mask = (trades_df['transactionDate'] >= pd.to_datetime(start_range)) & \
           (trades_df['transactionDate'] <= pd.to_datetime(end_range))
    trades_df = trades_df.loc[mask]

    if trades_df.empty:
        print(f"No senator trades with transactionDate between {start_range} and {end_range} for {ticker}.")
        return pd.DataFrame()

    trades_df['senator'] = trades_df['firstName'] + ' ' + trades_df['lastName']
    result_rows = []

    for _, row in trades_df.iterrows():
        trans_date = row['transactionDate']
        received_date = row['dateRecieved']
        senator = row['senator']

        # Alım günü fiyatı
        start_td = (trans_date - timedelta(days=3)).strftime('%Y-%m-%d')
        end_td   = (trans_date + timedelta(days=3)).strftime('%Y-%m-%d')
        trans_price_series = fmp_close_series(ticker, start_td, end_td)
        if trans_price_series.empty:
            continue
        idx_trans = trans_price_series.index.get_indexer([trans_date], method="nearest")[0]
        transaction_price = trans_price_series.iloc[idx_trans]

        # Rapor günü fiyatı
        start_rd = (received_date - timedelta(days=3)).strftime('%Y-%m-%d')
        end_rd   = (received_date + timedelta(days=3)).strftime('%Y-%m-%d')
        received_price_series = fmp_close_series(ticker, start_rd, end_rd)
        if received_price_series.empty:
            continue
        idx_rec = received_price_series.index.get_indexer([received_date], method="nearest")[0]
        received_price = received_price_series.iloc[idx_rec]

        # Farklar
        diff = received_price - transaction_price
        diff_pct = (diff / transaction_price) * 100

        result_rows.append({
            "Senator": senator,
            "Transaction Date": trans_date.date(),
            "Received Date": received_date.date(),
            "Transaction Price": round(transaction_price, 2),
            "Received Price": round(received_price, 2),
            "Price Diff ($)": round(diff, 2),
            "Price Diff (%)": round(diff_pct, 2),
        })

        time.sleep(0.25)  # API koruması

    result_df = pd.DataFrame(result_rows)
    return result_df


In [31]:
df = compare_trade_vs_received_price("AAPL", "2012-01-01", "2024-12-31")
print(df.head())


                Senator Transaction Date Received Date  Transaction Price  \
0      Tommy Tuberville       2024-01-10    2024-02-13             186.19   
1      Markwayne Mullin       2024-01-02    2024-01-25             185.64   
2      Markwayne Mullin       2023-10-03    2023-10-31             172.40   
3      Markwayne Mullin       2023-09-13    2023-10-11             174.21   
4  Shelley Moore Capito       2023-06-14    2023-07-16             183.95   

   Received Price  Price Diff ($)  Price Diff (%)  
0          185.04           -1.15           -0.62  
1          194.17            8.53            4.59  
2          170.77           -1.63           -0.95  
3          179.80            5.59            3.21  
4          193.99           10.04            5.46  


In [32]:
df

,Senator,Transaction Date,Received Date,Transaction Price,Received Price,Price Diff ($),Price Diff (%)
0,Tommy Tuberville,2024-01-10,2024-02-13,186.19,185.04,-1.15,-0.62
1,Markwayne Mullin,2024-01-02,2024-01-25,185.64,194.17,8.53,4.59
2,Markwayne Mullin,2023-10-03,2023-10-31,172.40,170.77,-1.63,-0.95
3,Markwayne Mullin,2023-09-13,2023-10-11,174.21,179.80,5.59,3.21
4,Shelley Moore Capito,2023-06-14,2023-07-16,183.95,193.99,10.04,5.46
...,...,...,...,...,...,...,...
134,Pat Roberts,2014-09-12,2014-10-06,25.42,24.91,-0.51,-2.01
135,Pat Roberts,2014-07-30,2014-08-12,24.54,23.99,-0.55,-2.24
136,Pat Roberts,2014-08-01,2014-08-12,24.03,23.99,-0.04,-0.17
137,John Reed,2014-03-28,2014-04-09,19.17,18.94,-0.23,-1.20


In [ ]:
https://financialmodelingprep.com/api/v3/etf-holder/SPY?apikey=YOUR_API_KEY
# Yapılacaklar
# Eski sistemdeki deterministik metrikler kalıcak
# Top 20yi seçmiycek bir puanlama üstünü alacak oturmuş bir puanlama
# Senatör checki yapılcak satması takip edilecek yıl bazlı işlem değil satış tarihi değişebilecek
# Senatör checki alımlarda da pozitif etkenli bir faz olucak ranking sonrası
# Senatör puanka üstteki ve sell ile
# Meclis temsilcilerini de ekle
# Etl fazı da ekle
# Fraud detection negatif değişkenli bir faz olacak
# Parametrizasyon güncellemesi için YK şeyleri kullanılacak optuna


In [ ]:
# Senatör Featureunda yapmak istediğim:
# Senatörlerin rankingini sağlamak bu 2 farklı yolun birleşmesiyle olucak
# Biri senator trust index diyebiliriz bu geçmişe yönelik işlemlerde
# Bunda 2 spektrum yaparız buy farklı sell farklı olur 
# Transaction date ve received datedeki fiyatları karşılaştıracak
# Bu sayede aradaki sürede fiyat farkı az mı olmuş uzun vadeli mi
# Kısa vadeli mi yatırım yapıyor görüyor olacağız
# İkincisi senatör performans endeksi backtestle beraber
# Bir nevi senatörlerin performanslarını yorumlayacağız
# Bu elde ettiğimiz rankingi kullanacğımız bir faz olucak
# Ekonomik metriklerle sıraladığımız ve belli bir değerin üstünü yatırım
# için değerlendirmeye geçmeden yakın zamanda senatörler kendi
# rating katsayılarıyla beraber buy pozitif etki edicek
# sell negatif etki edicek sonrasında yine ranking hisselerimiz olacak

# Ayrıca yapılacaklar: 
# Otomatik power bi raporları
# JupyterHUBA geçiş ya da benzeri
# Trade box enjeksityonu
# Zipline a bak https://zipline.ml4trading.io



In [1]:
import requests
import pandas as pd
from datetime import datetime

# Sembol seçimi
symbol = "AAPL"

# API anahtarı
API_KEY = open("FMP API KEY.txt").read().strip()

# API endpoint
url = f"https://financialmodelingprep.com/api/v4/senate-trading?symbol={symbol}&apikey={API_KEY}"

# API'den veri çek
response = requests.get(url)
data = response.json()

# DataFrame oluştur
df = pd.DataFrame(data)

# Tarih formatını dönüştür
df['transactionDate'] = pd.to_datetime(df['transactionDate'])

# En güncel işlemler en üstte olacak şekilde sırala
df.sort_values(by='transactionDate', ascending=False, inplace=True)

# Gereksiz kolonları at (isteğe bağlı)
columns_to_keep = ['transactionDate', 'senator', 'assetDescription', 'type', 'amount', 'owner', 'ticker']
df = df[columns_to_keep]

# Temel görüntüleme
print(df.head(10))


KeyError: "['senator', 'ticker'] not in index"

In [2]:
#!/usr/bin/env python
import time, json, certifi
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import ssl
from urllib.request import urlopen

# API ayarları
API_KEY   = open("FMP API KEY.txt").read().strip()
BASE_URL  = "https://financialmodelingprep.com/api/v3/"

# SSL güvenli veri çekme
def get_jsonparsed_data(url: str):
    context = ssl.create_default_context(cafile=certifi.where())
    with urlopen(url, context=context) as resp:
        return json.loads(resp.read().decode("utf-8"))

# Kapanış fiyat serisi getir
def fmp_close_series(ticker: str, start: str, end: str) -> pd.Series:
    url = f"{BASE_URL}historical-price-full/{ticker}?from={start}&to={end}&apikey={API_KEY}"
    js = get_jsonparsed_data(url)
    if not js or "historical" not in js or not js["historical"]:
        return pd.Series()
    df = pd.DataFrame(js["historical"])
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date").sort_index()
    return df["close"]

# Alım işlemi analizi
def compare_trade_vs_received_price(ticker: str, start_range: str, end_range: str) -> pd.DataFrame:
    url = f"https://financialmodelingprep.com/api/v4/senate-trading?symbol={ticker}&apikey={API_KEY}"
    data = get_jsonparsed_data(url)
    trades_df = pd.DataFrame(data)

    if trades_df.empty:
        print(f"No senator trades found for {ticker}.")
        return pd.DataFrame()

    trades_df['transactionDate'] = pd.to_datetime(trades_df['transactionDate'])
    trades_df['dateRecieved'] = pd.to_datetime(trades_df['dateRecieved'])

    # Sadece alım işlemleri
    trades_df = trades_df[trades_df['type'].str.lower().isin(['purchase', 'buy'])]

    # Tarih aralığında filtrele
    mask = (trades_df['transactionDate'] >= pd.to_datetime(start_range)) & \
           (trades_df['transactionDate'] <= pd.to_datetime(end_range))
    trades_df = trades_df.loc[mask]

    if trades_df.empty:
        print(f"No senator trades between {start_range} and {end_range}.")
        return pd.DataFrame()

    trades_df['senator'] = trades_df['firstName'] + ' ' + trades_df['lastName']
    result_rows = []

    for _, row in trades_df.iterrows():
        trans_date = row['transactionDate']
        received_date = row['dateRecieved']
        senator = row['senator']

        # Alım fiyatı
        trans_price_series = fmp_close_series(ticker, (trans_date - timedelta(days=3)).strftime('%Y-%m-%d'),
                                                       (trans_date + timedelta(days=3)).strftime('%Y-%m-%d'))
        if trans_price_series.empty:
            continue
        idx_trans = trans_price_series.index.get_indexer([trans_date], method="nearest")[0]
        transaction_price = trans_price_series.iloc[idx_trans]

        # Rapor tarihi fiyatı
        received_price_series = fmp_close_series(ticker, (received_date - timedelta(days=3)).strftime('%Y-%m-%d'),
                                                           (received_date + timedelta(days=3)).strftime('%Y-%m-%d'))
        if received_price_series.empty:
            continue
        idx_rec = received_price_series.index.get_indexer([received_date], method="nearest")[0]
        received_price = received_price_series.iloc[idx_rec]

        diff = received_price - transaction_price
        diff_pct = (diff / transaction_price) * 100
        holding_days = (received_date - trans_date).days
        investor_type = "Short-Term" if holding_days <= 14 else "Long-Term"

        result_rows.append({
            "Senator": senator,
            "Transaction Date": trans_date.date(),
            "Received Date": received_date.date(),
            "Holding Days": holding_days,
            "Investor Type": investor_type,
            "Transaction Price": round(transaction_price, 2),
            "Received Price": round(received_price, 2),
            "Price Diff ($)": round(diff, 2),
            "Price Diff (%)": round(diff_pct, 2),
        })

        time.sleep(0.2)  # API koruması

    return pd.DataFrame(result_rows)

# Senatör bazlı özet performans tablosu
def summarize_senators(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    
    summary = df.groupby("Senator").agg({
        "Price Diff ($)": "sum",
        "Price Diff (%)": "mean",
        "Investor Type": lambda x: x.mode()[0],
        "Transaction Date": "count"
    }).rename(columns={
        "Price Diff ($)": "Total Gains ($)",
        "Price Diff (%)": "Avg Gain (%)",
        "Transaction Date": "Number of Trades"
    }).reset_index()

    return summary

# Kullanım örneği
if __name__ == "__main__":
    ticker = "AAPL"
    start = "2022-01-01"
    end = "2024-12-31"

    df_trades = compare_trade_vs_received_price(ticker, start, end)
    print("\n🔍 Trade Bazlı Detaylar:\n", df_trades)

    df_summary = summarize_senators(df_trades)
    print("\n📊 Senatör Performans Özeti:\n", df_summary)



🔍 Trade Bazlı Detaylar:
                 Senator Transaction Date Received Date  Holding Days  \
0      Tommy Tuberville       2024-01-10    2024-02-13            34   
1      Markwayne Mullin       2024-01-02    2024-01-25            23   
2      Markwayne Mullin       2023-10-03    2023-10-31            28   
3      Markwayne Mullin       2023-09-13    2023-10-11            28   
4  Shelley Moore Capito       2023-06-14    2023-07-16            32   
5          Dan Sullivan       2022-11-22    2022-12-21            29   
6             Ron Wyden       2022-02-24    2022-03-23            27   
7             Ron Wyden       2022-02-24    2022-03-22            26   

  Investor Type  Transaction Price  Received Price  Price Diff ($)  \
0     Long-Term             186.19          185.04           -1.15   
1     Long-Term             185.64          194.17            8.53   
2     Long-Term             172.40          170.77           -1.63   
3     Long-Term             174.21          1

In [3]:
df_summary

,Senator,Total Gains ($),Avg Gain (%),Investor Type,Number of Trades
0,Dan Sullivan,-14.73,-9.810000,Long-Term,1
1,Markwayne Mullin,12.49,2.283333,Long-Term,3
2,Ron Wyden,13.55,4.165000,Long-Term,2
3,Shelley Moore Capito,10.04,5.460000,Long-Term,1
4,Tommy Tuberville,-1.15,-0.620000,Long-Term,1


In [5]:
import time, json, certifi
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import ssl
from urllib.request import urlopen

API_KEY   = open("FMP API KEY.txt").read().strip()
BASE_URL  = "https://financialmodelingprep.com/api/v3/"

# Veri çekme
def get_jsonparsed_data(url: str):
    context = ssl.create_default_context(cafile=certifi.where())
    with urlopen(url, context=context) as resp:
        return json.loads(resp.read().decode("utf-8"))

# Fiyat serisi
def fmp_close_series(ticker: str, start: str, end: str) -> pd.Series:
    url = f"{BASE_URL}historical-price-full/{ticker}?from={start}&to={end}&apikey={API_KEY}"
    js = get_jsonparsed_data(url)
    if not js or "historical" not in js or not js["historical"]:
        return pd.Series()
    df = pd.DataFrame(js["historical"])
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date").sort_index()
    return df["close"]

# Senatör işlem analizi
def compare_trade_vs_received_price(ticker: str, start_range: str, end_range: str) -> pd.DataFrame:
    url = f"https://financialmodelingprep.com/api/v4/senate-trading?symbol={ticker}&apikey={API_KEY}"
    try:
        data = get_jsonparsed_data(url)
    except:
        return pd.DataFrame()
    
    trades_df = pd.DataFrame(data)
    if trades_df.empty:
        return pd.DataFrame()

    trades_df['transactionDate'] = pd.to_datetime(trades_df['transactionDate'])
    trades_df['dateRecieved'] = pd.to_datetime(trades_df['dateRecieved'])
    trades_df = trades_df[trades_df['type'].str.lower().isin(['purchase', 'buy'])]

    mask = (trades_df['transactionDate'] >= pd.to_datetime(start_range)) & \
           (trades_df['transactionDate'] <= pd.to_datetime(end_range))
    trades_df = trades_df.loc[mask]

    if trades_df.empty:
        return pd.DataFrame()

    trades_df['senator'] = trades_df['firstName'] + ' ' + trades_df['lastName']
    result_rows = []

    for _, row in trades_df.iterrows():
        trans_date = row['transactionDate']
        received_date = row['dateRecieved']
        senator = row['senator']

        trans_price_series = fmp_close_series(ticker, (trans_date - timedelta(days=3)).strftime('%Y-%m-%d'),
                                                       (trans_date + timedelta(days=3)).strftime('%Y-%m-%d'))
        if trans_price_series.empty:
            continue
        idx_trans = trans_price_series.index.get_indexer([trans_date], method="nearest")[0]
        transaction_price = trans_price_series.iloc[idx_trans]

        received_price_series = fmp_close_series(ticker, (received_date - timedelta(days=3)).strftime('%Y-%m-%d'),
                                                           (received_date + timedelta(days=3)).strftime('%Y-%m-%d'))
        if received_price_series.empty:
            continue
        idx_rec = received_price_series.index.get_indexer([received_date], method="nearest")[0]
        received_price = received_price_series.iloc[idx_rec]

        diff = received_price - transaction_price
        diff_pct = (diff / transaction_price) * 100
        holding_days = (received_date - trans_date).days
        investor_type = "Short-Term" if holding_days <= 14 else "Long-Term"

        result_rows.append({
            "Senator": senator,
            "Ticker": ticker,
            "Transaction Date": trans_date.date(),
            "Received Date": received_date.date(),
            "Holding Days": holding_days,
            "Investor Type": investor_type,
            "Transaction Price": round(transaction_price, 2),
            "Received Price": round(received_price, 2),
            "Price Diff ($)": round(diff, 2),
            "Price Diff (%)": round(diff_pct, 2),
        })

        time.sleep(0.2)  # API koruması

    return pd.DataFrame(result_rows)

# S&P 500 şirketlerini al
def get_sp500_tickers() -> list:
    url = f"{BASE_URL}sp500_constituent?apikey={API_KEY}"
    data = get_jsonparsed_data(url)
    return [item['symbol'] for item in data]

# Özet analiz
def summarize_senators(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    summary = df.groupby("Senator").agg({
        "Price Diff ($)": "sum",
        "Price Diff (%)": "mean",
        "Investor Type": lambda x: x.mode()[0],
        "Transaction Date": "count"
    }).rename(columns={
        "Price Diff ($)": "Total Gains ($)",
        "Price Diff (%)": "Avg Gain (%)",
        "Transaction Date": "Number of Trades"
    }).reset_index()
    return summary

# Çalıştırıcı
if __name__ == "__main__":
    start = "2022-01-01"
    end = "2024-12-31"
    all_trades = []

    tickers = get_sp500_tickers()
    for i, ticker in enumerate(tickers):
        print(f"[{i+1}/{len(tickers)}] Analyzing {ticker}...")
        df = compare_trade_vs_received_price(ticker, start, end)
        if not df.empty:
            all_trades.append(df)

    combined_df = pd.concat(all_trades, ignore_index=True)
    combined_df.to_csv("senator_trades_sp500.csv", index=False)

    summary_df = summarize_senators(combined_df)
    summary_df.to_csv("senator_summary_sp500.csv", index=False)

    print("✅ Tüm analiz tamamlandı. CSV dosyaları oluşturuldu.")


[1/503] Analyzing COIN...
[2/503] Analyzing DASH...
[3/503] Analyzing EXE...
[4/503] Analyzing TKO...
[5/503] Analyzing WSM...
[6/503] Analyzing APO...
[7/503] Analyzing LII...
[8/503] Analyzing WDAY...
[9/503] Analyzing TPL...
[10/503] Analyzing DELL...
[11/503] Analyzing ERIE...
[12/503] Analyzing PLTR...
[13/503] Analyzing SW...
[14/503] Analyzing CRWD...
[15/503] Analyzing GDDY...
[16/503] Analyzing KKR...
[17/503] Analyzing VST...
[18/503] Analyzing GEV...
[19/503] Analyzing SOLV...
[20/503] Analyzing DECK...
[21/503] Analyzing SMCI...
[22/503] Analyzing BLDR...
[23/503] Analyzing JBL...
[24/503] Analyzing UBER...
[25/503] Analyzing HUBB...
[26/503] Analyzing LULU...
[27/503] Analyzing VLTO...
[28/503] Analyzing ABNB...
[29/503] Analyzing BX...
[30/503] Analyzing KVUE...
[31/503] Analyzing PANW...
[32/503] Analyzing AXON...
[33/503] Analyzing FICO...
[34/503] Analyzing BG...
[35/503] Analyzing PODD...
[36/503] Analyzing GEHC...
[37/503] Analyzing STLD...
[38/503] Analyzing FSLR...

In [4]:
import requests
import pandas as pd

# Senin API anahtarını buraya koy
API_KEY   = open("FMP API KEY.txt").read().strip()
etf_symbol = "SPY"

# API URL
url = f"https://financialmodelingprep.com/api/v4/etf-info?symbol={etf_symbol}&apikey={API_KEY}"

# API'den veri çek
response = requests.get(url)
if response.status_code != 200:
    raise Exception(f"API error: {response.status_code}")
data = response.json()

# Sektör dağılımını ayıkla
if data and 'sectorExposure' in data[0]:
    sectors = data[0]['sectorExposure']
    df = pd.DataFrame(sectors)
    df.columns = ['Sector', 'Weight (%)']
    print(df)
else:
    print("Sector data not found for the ETF.")


Sector data not found for the ETF.
